# Agentic Weekly Learning Insights

In this notebook, I build a simple agentic workflow that reads weekly study behavior data and generates personalized learning insights.

The goal is to convert numerical study patterns into plain-language recommendations.

Instead of only showing charts or model outputs, this workflow summarizes the learner's weekly behavior and suggests one specific improvement for the next week.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [18]:
df = pd.read_csv("../data/processed/cleaned_study_sessions.csv")

df.head()

,session_id,date,day_of_week,start_time,end_time,hour_of_day,time_of_day,duration_min,topic,topic_type,difficulty_1_5,sleep_hours,energy_before_1_10,distraction_level_1_10,break_before_days,location,device_used,completion_percent,focus_score_1_10,notes,year,month,week,weekday_number
0,S0001,2024-01-08,Monday,15:10,16:10,15,Afternoon,60,Machine Learning,ML,4,7.8,6,2,0,Study Room,Laptop + Notebook,77,6,"Decent session, some distractions but useful p...",2024,1,2,0
1,S0002,2024-01-09,Tuesday,16:00,16:50,16,Afternoon,50,Python Programming,Programming,3,7.5,6,5,0,Study Room,Laptop,68,5,"Low energy, needed more breaks than expected",2024,1,2,1
2,S0003,2024-01-10,Wednesday,18:15,19:15,18,Evening,60,German Language,Language,3,7.3,5,2,0,Cafe,Laptop + Notebook,90,6,"Decent session, some distractions but useful p...",2024,1,2,2
3,S0004,2024-01-11,Thursday,15:45,16:30,15,Afternoon,45,Linear Algebra,Math,4,7.2,7,4,0,Home Desk,Laptop + Notebook,62,5,"Low energy, needed more breaks than expected",2024,1,2,3
4,S0005,2024-01-12,Friday,10:00,11:00,10,Morning,60,SQL Practice,Programming,3,8.5,7,3,0,University Library,Laptop + Notebook,91,8,"High clarity, strong concentration, completed ...",2024,1,2,4


In [19]:
df["date"] = pd.to_datetime(df["date"])

df["date"].min(), df["date"].max()

(Timestamp('2024-01-08 00:00:00'), Timestamp('2025-12-28 00:00:00'))

In [20]:
weekly_summary = (
    df.set_index("date")
    .resample("W")
    .agg(
        sessions_count=("session_id", "count"),
        avg_focus_score=("focus_score_1_10", "mean"),
        avg_duration_min=("duration_min", "mean"),
        avg_sleep_hours=("sleep_hours", "mean"),
        avg_energy_before=("energy_before_1_10", "mean"),
        avg_distraction_level=("distraction_level_1_10", "mean"),
        avg_completion_percent=("completion_percent", "mean"),
        total_study_minutes=("duration_min", "sum"),
        avg_break_before_days=("break_before_days", "mean")
    )
    .reset_index()
)

weekly_summary.head()

,date,sessions_count,avg_focus_score,avg_duration_min,avg_sleep_hours,avg_energy_before,avg_distraction_level,avg_completion_percent,total_study_minutes,avg_break_before_days
0,2024-01-14,8,5.500000,70.000000,7.637500,6.125000,3.375000,79.000000,560,0.000000
1,2024-01-21,8,5.750000,56.250000,6.675000,5.000000,3.125000,85.250000,450,0.250000
2,2024-01-28,3,5.666667,66.666667,6.633333,6.666667,2.666667,73.333333,200,1.333333
3,2024-02-04,3,4.333333,66.666667,7.200000,5.333333,4.333333,71.000000,200,1.333333
4,2024-02-11,6,6.333333,100.000000,7.083333,6.500000,3.666667,90.000000,600,0.166667


In [21]:
print("Weekly summary shape:", weekly_summary.shape)

weekly_summary.tail(10)

Weekly summary shape: (103, 10)


,date,sessions_count,avg_focus_score,avg_duration_min,avg_sleep_hours,avg_energy_before,avg_distraction_level,avg_completion_percent,total_study_minutes,avg_break_before_days
93,2025-10-26,6,6.500000,83.333333,6.866667,6.166667,3.000000,81.833333,500,0.333333
94,2025-11-02,9,6.000000,78.888889,6.722222,6.555556,3.777778,85.111111,710,0.000000
95,2025-11-09,4,5.000000,82.500000,7.675000,5.000000,4.250000,80.750000,330,1.000000
96,2025-11-16,3,7.333333,75.000000,6.666667,7.333333,2.666667,82.000000,225,1.333333
97,2025-11-23,6,4.833333,70.833333,7.450000,5.833333,4.166667,70.500000,425,0.666667
98,2025-11-30,9,6.222222,71.111111,7.055556,5.666667,3.000000,88.666667,640,0.111111
99,2025-12-07,9,6.222222,80.000000,7.266667,6.555556,2.666667,85.111111,720,0.222222
100,2025-12-14,5,6.400000,93.000000,7.620000,6.400000,3.800000,86.200000,465,0.400000
101,2025-12-21,5,6.400000,93.000000,6.480000,6.400000,3.200000,89.000000,465,0.600000
102,2025-12-28,6,6.000000,62.500000,7.050000,5.666667,3.333333,80.166667,375,0.333333


In [22]:
import os

# To run for a specific 7-day period ending on a date, set target_date to a date string (YYYY-MM-DD)
# e.g., target_date = "2025-02-07" or "2025-12-21"
# If None, it defaults to the latest date available in the dataset.
target_date = os.getenv("INSIGHT_WEEK_DATE", None)

if target_date:
    try:
        target_dt = pd.to_datetime(target_date)
    except Exception as e:
        print(f"Error parsing date '{target_date}': {e}. Defaulting to latest date.")
        target_dt = df["date"].max()
else:
    target_dt = df["date"].max()

start_dt = target_dt - pd.Timedelta(days=6)
print(f"Generating 7-day summary from {start_dt.date()} to {target_dt.date()}...")

# Filter the original daily sessions dataframe
sub_df = df[(df["date"] >= start_dt) & (df["date"] <= target_dt)]

# Compute the summary stats dynamically for the past 7 days
summary_stats = {
    "date": target_dt,
    "sessions_count": sub_df["session_id"].count(),
    "avg_focus_score": sub_df["focus_score_1_10"].mean() if not sub_df.empty else 0.0,
    "avg_duration_min": sub_df["duration_min"].mean() if not sub_df.empty else 0.0,
    "avg_sleep_hours": sub_df["sleep_hours"].mean() if not sub_df.empty else 0.0,
    "avg_energy_before": sub_df["energy_before_1_10"].mean() if not sub_df.empty else 0.0,
    "avg_distraction_level": sub_df["distraction_level_1_10"].mean() if not sub_df.empty else 0.0,
    "avg_completion_percent": sub_df["completion_percent"].mean() if not sub_df.empty else 0.0,
    "total_study_minutes": sub_df["duration_min"].sum(),
    "avg_break_before_days": sub_df["break_before_days"].mean() if not sub_df.empty else 0.0
}
selected_week = pd.Series(summary_stats)
selected_week

Generating 7-day summary from 2024-09-18 to 2024-09-24...


date                      2024-09-24 00:00:00
sessions_count                              6
avg_focus_score                           6.0
avg_duration_min                    93.333333
avg_sleep_hours                      7.266667
avg_energy_before                    6.833333
avg_distraction_level                5.333333
avg_completion_percent              82.833333
total_study_minutes                       560
avg_break_before_days                     0.5
dtype: object

In [23]:
week_text = f"""
Weekly Learning Summary

Week ending: {selected_week['date'].date()}

Number of study sessions: {selected_week['sessions_count']}
Average focus score: {selected_week['avg_focus_score']:.2f}/10
Average session duration: {selected_week['avg_duration_min']:.1f} minutes
Average sleep hours: {selected_week['avg_sleep_hours']:.1f}
Average energy before studying: {selected_week['avg_energy_before']:.2f}/10
Average distraction level: {selected_week['avg_distraction_level']:.2f}/10
Average completion percentage: {selected_week['avg_completion_percent']:.1f}%
Total study time: {selected_week['total_study_minutes']:.0f} minutes
Average break before sessions: {selected_week['avg_break_before_days']:.2f} days
"""

print(week_text)


Weekly Learning Summary

Week ending: 2024-09-24

Number of study sessions: 6
Average focus score: 6.00/10
Average session duration: 93.3 minutes
Average sleep hours: 7.3
Average energy before studying: 6.83/10
Average distraction level: 5.33/10
Average completion percentage: 82.8%
Total study time: 560 minutes
Average break before sessions: 0.50 days



In [24]:
def generate_basic_weekly_insight(week):
    insights = []

    if week["avg_focus_score"] >= 7:
        insights.append("Focus was strong this week. The learner maintained good concentration during study sessions.")
    elif week["avg_focus_score"] >= 5:
        insights.append("Focus was moderate this week. There is room to improve consistency and session quality.")
    else:
        insights.append("Focus was low this week. The learner may need to adjust study timing, reduce distractions, or improve rest.")

    if week["avg_distraction_level"] > 6:
        insights.append("Distraction level was high, which may have reduced focus and completion percentage.")
    
    if week["avg_sleep_hours"] < 6.5:
        insights.append("Average sleep was below the recommended level, which may have affected attention and energy.")

    if week["avg_completion_percent"] < 70:
        insights.append("Completion percentage was relatively low, meaning planned study sessions were not fully completed.")

    if week["sessions_count"] < 3:
        insights.append("The number of study sessions was low, which may indicate a consistency gap.")

    recommendation = "Recommended action: Schedule at least one focused morning study session next week and reduce distractions before starting."

    return "\n".join(insights) + "\n\n" + recommendation


basic_insight = generate_basic_weekly_insight(selected_week)

print(basic_insight)

Focus was moderate this week. There is room to improve consistency and session quality.

Recommended action: Schedule at least one focused morning study session next week and reduce distractions before starting.


In [25]:
import os

os.makedirs("../outputs/weekly_insights", exist_ok=True)

week_date_str = selected_week['date'].date()
file_path = f"../outputs/weekly_insights/basic_weekly_insight_{week_date_str}.txt"

with open(file_path, "w", encoding="utf-8") as file:
    file.write(week_text)
    file.write("\n")
    file.write(basic_insight)

print(f"Basic weekly insight saved successfully to {file_path}")

Basic weekly insight saved successfully to ../outputs/weekly_insights/basic_weekly_insight_2024-09-24.txt


## LangChain + OpenAI Weekly Insight Workflow

The previous section created a basic rule-based weekly insight generator.

In this section, I add an LLM-based workflow using LangChain and the OpenAI API. This converts weekly study behavior metrics into a more natural, personalized learning summary and one specific recommendation.

The workflow is designed so the notebook can still run safely even when an API key is not available.

In [26]:
import sys

!{sys.executable} -m pip install python-dotenv langchain langchain-openai openai

Defaulting to user installation because normal site-packages is not writeable


In [27]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    
    langchain_available = True
    print("LangChain and OpenAI packages loaded successfully.")
except ImportError as e:
    langchain_available = False
    print("LangChain/OpenAI packages are not installed.")
    print(e)

LangChain and OpenAI packages loaded successfully.


In [28]:
prompt = ChatPromptTemplate.from_template("""
You are a personal learning analytics assistant.

You will receive a weekly study behavior summary.

Your task:
1. Summarize the week in plain language.
2. Identify the main pattern.
3. Suggest exactly one specific adjustment for next week.
4. Keep the answer practical and not generic.

Weekly data:
{weekly_summary}

Return the answer in this format:

Weekly Summary:
...

Main Pattern:
...

Recommended Adjustment:
...
""")

print("Prompt template created successfully.")

Prompt template created successfully.


In [29]:
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_api_base = os.getenv("OPENAI_API_BASE")
openai_model_name = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

if openai_api_key and langchain_available:
    llm_kwargs = {
        "model": openai_model_name,
        "openai_api_key": openai_api_key,
        "temperature": 0.3
    }
    if openai_api_base:
        llm_kwargs["base_url"] = openai_api_base
        
    llm = ChatOpenAI(**llm_kwargs)

    chain = prompt | llm | StrOutputParser()

    print(f"LangChain chain created successfully using model: {openai_model_name}")
else:
    chain = None
    print("OpenAI API key not found. LangChain chain was not created.")
    print("The notebook can still run using the rule-based insight generator.")

LangChain chain created successfully using model: openai/gpt-oss-120b:free


In [30]:
# Full recovery cell: reload data, recreate weekly_summary, selected_week, week_text, and basic_insight

import pandas as pd
import os

df = pd.read_csv("../data/processed/cleaned_study_sessions.csv")
df["date"] = pd.to_datetime(df["date"])

weekly_summary = (
    df.set_index("date")
    .resample("W")
    .agg(
        sessions_count=("session_id", "count"),
        avg_focus_score=("focus_score_1_10", "mean"),
        avg_duration_min=("duration_min", "mean"),
        avg_sleep_hours=("sleep_hours", "mean"),
        avg_energy_before=("energy_before_1_10", "mean"),
        avg_distraction_level=("distraction_level_1_10", "mean"),
        avg_completion_percent=("completion_percent", "mean"),
        total_study_minutes=("duration_min", "sum"),
        avg_break_before_days=("break_before_days", "mean")
    )
    .reset_index()
)

target_date = os.getenv("INSIGHT_WEEK_DATE", None)
if target_date:
    try:
        target_dt = pd.to_datetime(target_date)
    except Exception:
        target_dt = df["date"].max()
else:
    target_dt = df["date"].max()

start_dt = target_dt - pd.Timedelta(days=6)
sub_df = df[(df["date"] >= start_dt) & (df["date"] <= target_dt)]

summary_stats = {
    "date": target_dt,
    "sessions_count": sub_df["session_id"].count(),
    "avg_focus_score": sub_df["focus_score_1_10"].mean() if not sub_df.empty else 0.0,
    "avg_duration_min": sub_df["duration_min"].mean() if not sub_df.empty else 0.0,
    "avg_sleep_hours": sub_df["sleep_hours"].mean() if not sub_df.empty else 0.0,
    "avg_energy_before": sub_df["energy_before_1_10"].mean() if not sub_df.empty else 0.0,
    "avg_distraction_level": sub_df["distraction_level_1_10"].mean() if not sub_df.empty else 0.0,
    "avg_completion_percent": sub_df["completion_percent"].mean() if not sub_df.empty else 0.0,
    "total_study_minutes": sub_df["duration_min"].sum(),
    "avg_break_before_days": sub_df["break_before_days"].mean() if not sub_df.empty else 0.0
}
selected_week = pd.Series(summary_stats)

week_text = f"""
Weekly Learning Summary

Week ending: {selected_week['date'].date()}

Number of study sessions: {selected_week['sessions_count']}
Average focus score: {selected_week['avg_focus_score']:.2f}/10
Average session duration: {selected_week['avg_duration_min']:.1f} minutes
Average sleep hours: {selected_week['avg_sleep_hours']:.1f}
Average energy before studying: {selected_week['avg_energy_before']:.2f}/10
Average distraction level: {selected_week['avg_distraction_level']:.2f}/10
Average completion percentage: {selected_week['avg_completion_percent']:.1f}%
Total study time: {selected_week['total_study_minutes']:.0f} minutes
Average break before sessions: {selected_week['avg_break_before_days']:.2f} days
"""

def generate_basic_weekly_insight(week):
    insights = []

    if week["avg_focus_score"] >= 7:
        insights.append("Focus was strong this week. The learner maintained good concentration during study sessions.")
    elif week["avg_focus_score"] >= 5:
        insights.append("Focus was moderate this week. There is room to improve consistency and session quality.")
    else:
        insights.append("Focus was low this week. The learner may need to adjust study timing, reduce distractions, or improve rest.")

    if week["avg_distraction_level"] > 6:
        insights.append("Distraction level was high, which may have reduced focus and completion percentage.")

    if week["avg_sleep_hours"] < 6.5:
        insights.append("Average sleep was below the recommended level, which may have affected attention and energy.")

    if week["avg_completion_percent"] < 70:
        insights.append("Completion percentage was relatively low, meaning planned study sessions were not fully completed.")

    if week["sessions_count"] < 3:
        insights.append("The number of study sessions was low, which may indicate a consistency gap.")

    recommendation = "Recommended action: Schedule at least one focused morning study session next week and reduce distractions before starting."

    return "\n".join(insights) + "\n\n" + recommendation

basic_insight = generate_basic_weekly_insight(selected_week)

print("Full recovery completed successfully.")
print(week_text)
print(basic_insight)

Full recovery completed successfully.

Weekly Learning Summary

Week ending: 2025-10-24

Number of study sessions: 7
Average focus score: 5.29/10
Average session duration: 96.4 minutes
Average sleep hours: 6.8
Average energy before studying: 5.14/10
Average distraction level: 3.14/10
Average completion percentage: 83.1%
Total study time: 675 minutes
Average break before sessions: 0.29 days

Focus was moderate this week. There is room to improve consistency and session quality.

Recommended action: Schedule at least one focused morning study session next week and reduce distractions before starting.


In [31]:
if chain is not None:
    llm_weekly_insight = chain.invoke({
        "weekly_summary": week_text
    })

    print("LLM-based weekly insight generated:")
    print()
    print(llm_weekly_insight)

else:
    llm_weekly_insight = basic_insight

    print("Using rule-based insight because OpenAI API key is not available.")
    print()
    print(llm_weekly_insight)

LLM-based weekly insight generated:

**Weekly Summary:**  
You studied on 7 days, averaging about 1.6 hours per session for a total of roughly 11 hours of work. Your focus hovered around the middle of the scale (5.3/10) and you felt a similar level of energy (5.1/10) before each session. Distractions were relatively low (3.1/10), and you completed about 83 % of the planned material. Sleep averaged 6.8 hours per night, and you typically waited less than a third of a day (≈7 hours) after waking before you began studying.

**Main Pattern:**  
Your study time is consistent, but the modest focus and energy scores suggest you’re often starting sessions without feeling fully alert. The short gap between waking and beginning work (≈7 hours) means you’re likely beginning to study while still in a low‑energy state.

**Recommended Adjustment:**  
Add a 20‑minute “activation routine” right after you wake up—light stretching, a glass of water, and a brief brisk walk or set of jumping‑jacks—before y

In [32]:
os.makedirs("../outputs/weekly_insights", exist_ok=True)

week_date_str = selected_week['date'].date()
file_path = f"../outputs/weekly_insights/langchain_weekly_insight_{week_date_str}.txt"

with open(file_path, "w", encoding="utf-8") as file:
    file.write("LangChain/OpenAI Weekly Insight Workflow\n")
    file.write("=" * 45)
    file.write("\n\n")
    file.write(week_text)
    file.write("\n")
    file.write(llm_weekly_insight)

print(f"LangChain weekly insight output saved successfully to {file_path}")

LangChain weekly insight output saved successfully to ../outputs/weekly_insights/langchain_weekly_insight_2025-10-24.txt
